In [1]:
import os
import sys
import json
import numpy as np
import torch
from pathlib import Path
from skimage.io import imread
import napari
sys.path.append(os.path.abspath(".."))
import data
import models
from data import RangeAnnotationDataset
from models import AutomaticRangeNet


In [2]:
sample = "MF153_early_TMA1"

In [3]:
viewer = napari.Viewer()

In [ ]:
# List of markers to test
markers = ["CD4", "CD8a", "CD2", "FoxP3", "HLA-ABC", "CD163"]

# Load trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutomaticRangeNet().to(device)
model.load_state_dict(torch.load("../checkpoints/2_annotators_CD4_automatic_range.pt", map_location=device))
model.eval()

def load_img(path):
    img = imread(path)
    if img.max() > 1:
        if path.suffix in ['DAPI.tif', 'DAPI.tiff']:
            img = RangeAnnotationDataset.preprocess_data(img, percentile=99.99, remove_bright_artifacts=False)
        else :
            img = RangeAnnotationDataset.preprocess_data(img, percentile=99.9, remove_bright_artifacts=True, threshold_factor=2.5)
    return img

viewer = napari.Viewer()

for marker in markers:
    BATCH = f"generalization_CD4_to_{marker}"
    OUT_MARKER_DIR = Path(f"../data/{BATCH}/tiles_marker")
    OUT_DAPI_DIR = Path(f"../data/{BATCH}/tiles_DAPI")
    ANNOTATIONS_DIR = Path(f"../annotations/{BATCH}")

    # Get all marker tile files
    marker_tiles = sorted([f for f in OUT_MARKER_DIR.glob("*.tif*")])
    # Pick 2 random tiles for visualization
    np.random.seed(46)
    vis_tiles = np.random.choice(marker_tiles, size=min(2, len(marker_tiles)), replace=False)

    for tile_path in vis_tiles:
        tile_name = tile_path.stem
        marker_img = load_img(tile_path)
        dapi_path = OUT_DAPI_DIR / (tile_name.replace(marker, "DAPI") + tile_path.suffix)
        dapi_img = load_img(dapi_path)

        # Prepare input for model
        input_tensor = torch.from_numpy(np.stack([marker_img, dapi_img])[None, :, :, :]).float().to(device)
        with torch.no_grad():
            pred_min, pred_max = model(input_tensor).squeeze().tolist()

        # Try to get true annotation if available
        annotation_path = ANNOTATIONS_DIR / (tile_name + ".json")
        if annotation_path.exists():
            with open(annotation_path) as f:
                annotation = json.load(f)
            true_min, true_max = annotation["min"], annotation["max"]
        else:
            true_min, true_max = None, None

        # Add images to napari
        viewer.add_image(dapi_img, name=f"{marker} - {tile_name} DAPI", colormap='bop blue', contrast_limits=(0, 1))
        viewer.add_image(marker_img, name=f"{marker} - {tile_name} marker (pred)", colormap='gray', contrast_limits=(pred_min, pred_max), blending='additive')
        if true_min is not None and true_max is not None:
            viewer.add_image(marker_img, name=f"{marker} - {tile_name} marker (true)", colormap='green', contrast_limits=(true_min, true_max), blending='additive')



/tmp/ipykernel_20845/3062415163.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("../checkpoints/2_annotators_CD4_automatic_range.pt", ma